# EBI OLS4 Bulk Annotator

Resolve free-text strings to ontology **CURIEs** using the
[EMBL-EBI Ontology Lookup Service (OLS4)](https://www.ebi.ac.uk/ols4) REST API.

**No API key required.**

OLS4 `/search` endpoint is used — it returns ranked term matches scoped to a given ontology.
Results are collected into a pandas DataFrame and written to `ols4_annotations.csv` / `ols4_annotations.json`.

## 1. Setup

In [ ]:
import os
import re
import csv
import json
import time
import sys
from dataclasses import dataclass, field
from typing import Optional

import requests
import pandas as pd

OLS4_SEARCH_URL = "https://www.ebi.ac.uk/ols4/api/search"

# OLS4 is rate-limited; stay well under the threshold.
MAX_REQUESTS_PER_SECOND = 5
MIN_INTERVAL = 1.0 / MAX_REQUESTS_PER_SECOND

MAX_RETRIES = 5
INITIAL_BACKOFF = 2.0  # seconds

## 2. Helper functions

In [ ]:
@dataclass
class AnnotateItem:
    """One thing you want resolved to a CURIE."""
    text: str
    ontologies: list = field(default_factory=list)  # e.g. ["hp"] or ["uberon"] (lowercase for OLS4)
    label: Optional[str] = None                      # optional tag, e.g. "phenotype" / "taxon"
    exact_match: bool = True                         # True = exact query first, then falls back to fuzzy


def uri_to_curie(iri: str) -> str:
    """
    Convert an OBO-style IRI to a CURIE.

    http://purl.obolibrary.org/obo/HP_0000252   -> HP:0000252
    http://purl.obolibrary.org/obo/UBERON_0001155 -> UBERON:0001155
    Falls back to the raw IRI if no pattern matches.
    """
    m = re.search(r'/obo/([A-Za-z0-9]+)_([A-Za-z0-9]+)$', iri)
    if m:
        prefix, local_id = m.groups()
        return f"{prefix.upper()}:{local_id}"
    return iri

In [ ]:
def _call_ols4(text: str, ontologies: list, exact: bool) -> list:
    """Single throttled+retried call to OLS4 /search. Returns list of hit dicts."""
    params = {
        "q": text,
        "rows": 10,
        "lang": "en",
    }
    if ontologies:
        params["ontology"] = ",".join(o.lower() for o in ontologies)
    if exact:
        params["exact"] = "true"

    backoff = INITIAL_BACKOFF
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(OLS4_SEARCH_URL, params=params, timeout=30)
        except requests.RequestException as exc:
            sys.stderr.write(f"[warn] attempt {attempt}: request error: {exc}\n")
            time.sleep(backoff)
            backoff *= 2
            continue

        if resp.status_code == 200:
            data = resp.json()
            hits = data.get("response", {}).get("docs", [])
            # If exact returned nothing, retry without exact flag
            if not hits and exact:
                params.pop("exact", None)
                continue
            return hits

        if resp.status_code == 429:
            time.sleep(backoff)
            backoff *= 2
            continue

        sys.stderr.write(
            f"[warn] attempt {attempt}: HTTP {resp.status_code} for text={text!r}: "
            f"{resp.text[:200]}\n"
        )
        time.sleep(backoff)
        backoff *= 2

    sys.stderr.write(f"[error] giving up on text={text!r} after {MAX_RETRIES} attempts\n")
    return []


def bulk_annotate(items: list) -> list:
    """
    Run OLS4 search over a list of AnnotateItem.
    Returns a flat list of result dicts, one per (input item, matched hit).
    """
    results = []
    last_call_time = 0.0

    for item in items:
        elapsed = time.time() - last_call_time
        if elapsed < MIN_INTERVAL:
            time.sleep(MIN_INTERVAL - elapsed)

        hits = _call_ols4(item.text, item.ontologies, item.exact_match)
        last_call_time = time.time()

        if not hits:
            results.append({
                "input_text": item.text,
                "label": item.label,
                "matched_label": None,
                "curie": None,
                "iri": None,
                "ontology": None,
                "description": None,
            })
            continue

        for hit in hits:
            iri = hit.get("iri", "")
            curie = uri_to_curie(iri) if iri else None
            results.append({
                "input_text": item.text,
                "label": item.label,
                "matched_label": hit.get("label"),
                "curie": curie,
                "iri": iri,
                "ontology": hit.get("ontology_name"),
                "description": (hit.get("description") or [None])[0],
            })

    return results

## 3. Input items

Same inputs as the BioPortal notebook — HP, NCBITAXON, UBERON, MONDO — for a direct comparison.

In [ ]:
INPUT_ITEMS = [
    AnnotateItem(text="microcephaly", ontologies=["HP"], label="phenotype"),
    AnnotateItem(text="cleft palate", ontologies=["HP"], label="phenotype"),
    AnnotateItem(text="short stature", ontologies=["HP"], label="phenotype"),
    AnnotateItem(text="Escherichia coli", ontologies=["NCBITAXON"], label="taxon"),
    AnnotateItem(text="Homo sapiens", ontologies=["NCBITAXON"], label="taxon"),
    AnnotateItem(text="Mus musculus", ontologies=["NCBITAXON"], label="taxon"),
    # Body sites — UBERON
    AnnotateItem(text="colon", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="small intestine", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="cecum", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="rectum", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="oral cavity", ontologies=["UBERON"], label="body_site"),
    AnnotateItem(text="skin", ontologies=["UBERON"], label="body_site"),
    # Diseases — MONDO
    AnnotateItem(text="Crohn's disease", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="ulcerative colitis", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="colorectal cancer", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="type 2 diabetes", ontologies=["MONDO"], label="disease"),
    AnnotateItem(text="irritable bowel syndrome", ontologies=["MONDO"], label="disease"),
]

len(INPUT_ITEMS)

## 4. Run the bulk annotation

In [ ]:
results = bulk_annotate(INPUT_ITEMS)
df = pd.DataFrame(results)
df

## 5. Deduplicate — best match per input

OLS4 returns hits ranked by relevance; `first()` keeps the top match per input text.
Use `df` directly if you want all candidates.

In [ ]:
df_best = df.groupby("input_text", sort=False).first().reset_index()
df_best

## 6. Summary + save outputs

In [ ]:
matched = df_best["curie"].notna().sum()
print(f"{matched}/{len(df_best)} inputs resolved to a CURIE")

df_best.to_csv("ols4_annotations.csv", index=False)
with open("ols4_annotations.json", "w") as f:
    json.dump(results, f, indent=2)

print("Wrote ols4_annotations.csv and ols4_annotations.json")

## 7. Phenotype terms — MONDO vs HP comparison

Run each phenotype string against both ontologies and compare rank-1 results side by side.

In [10]:
PHENOTYPE_TERMS = [
    "microcephaly",
    "cleft palate",
    "short stature",
    "healthy",
    "acid reflux",
]

def compare_ontologies(terms: list[str], ont_a: str, ont_b: str) -> pd.DataFrame:
    """
    For each term, fetch rank-1 hit from ont_a and ont_b independently.
    Returns a side-by-side DataFrame.
    """
    rows = []
    for term in terms:
        hits_a = _call_ols4(term, [ont_a], exact=True) or _call_ols4(term, [ont_a], exact=False)
        hits_b = _call_ols4(term, [ont_b], exact=True) or _call_ols4(term, [ont_b], exact=False)

        top_a = hits_a[0] if hits_a else {}
        top_b = hits_b[0] if hits_b else {}

        iri_a = top_a.get("iri", "")
        iri_b = top_b.get("iri", "")

        rows.append({
            "term":               term,
            f"{ont_a}_curie":     uri_to_curie(iri_a) if iri_a else None,
            f"{ont_a}_label":     top_a.get("label"),
            f"{ont_a}_desc":      (top_a.get("description") or [None])[0],
            f"{ont_b}_curie":     uri_to_curie(iri_b) if iri_b else None,
            f"{ont_b}_label":     top_b.get("label"),
            f"{ont_b}_desc":      (top_b.get("description") or [None])[0],
        })
    return pd.DataFrame(rows)


df_phenotype = compare_ontologies(PHENOTYPE_TERMS, "MONDO", "HP")
df_phenotype

,term,MONDO_curie,MONDO_label,MONDO_desc,HP_curie,HP_label,HP_desc
0,microcephaly,MONDO:0001149,microcephaly,A congenital or acquired developmental disorde...,HP:0011451,Primary microcephaly,Head circumference below 2 standard deviations...
1,cleft palate,MONDO:0016064,cleft palate,Cleft palate is a fissure type embryopathy tha...,HP:0009099,Median cleft palate,Cleft palate of the midline of the palate.
2,short stature,MONDO:0014785,"microcephaly, short stature, and impaired gluc...","Any microcephaly, short stature, and impaired ...",HP:0003510,Severe short stature,The term severe short stature is to be preferr...
3,healthy,MONDO:0009194,immunodeficiency 32B,A rare progressive disease that begins as a pr...,HP:0001394,Cirrhosis,Cirrhosis is caused by chronic liver disease a...
4,acid reflux,MONDO:0007186,gastroesophageal reflux disease,A chronic disorder characterized by reflux of ...,HP:0002020,Gastroesophageal reflux,A condition in which the stomach contents leak...


## 8. Disease terms — MONDO vs HP comparison

Run each disease string against both ontologies and compare rank-1 results side by side.

In [11]:
DISEASE_TERMS = [
    "Crohn's disease",
    "ulcerative colitis",
    "colorectal cancer",
    "type 2 diabetes",
    "irritable bowel syndrome",
]

df_disease = compare_ontologies(DISEASE_TERMS, "MONDO", "HP")
df_disease

[error] giving up on text='irritable bowel syndrome' after 5 attempts


,term,MONDO_curie,MONDO_label,MONDO_desc,HP_curie,HP_label,HP_desc
0,Crohn's disease,MONDO:0005011,Crohn disease,A gastrointestinal disorder characterized by c...,HP:0100280,Crohn's disease,A chronic granulomatous inflammatory disease o...
1,ulcerative colitis,MONDO:0005101,ulcerative colitis,An inflammatory bowel disease involving the mu...,HP:0100279,Ulcerative colitis,A chronic inflammatory bowel disease that incl...
2,colorectal cancer,MONDO:0005575,colorectal cancer,Editor note: some sources make distinct classe...,HP:6000221,Positive stool occult blood test,This test is commonly used for colorectal canc...
3,type 2 diabetes,MONDO:0005148,type 2 diabetes mellitus,A type of diabetes mellitus that is characteri...,HP:0005978,Type II diabetes mellitus,Persons with type II diabetes mellitus rarely ...
4,irritable bowel syndrome,MONDO:0005052,irritable bowel syndrome,Irritable bowel syndrome (IBS) is a chronic fu...,None,None,None


## 9. Winner selection — MONDO-first strategy across all terms

Try MONDO first; if it resolves, use that CURIE and tag as `biolink:Disease`.
Fall back to HP and tag as `biolink:PhenotypicFeature`.
Combines both phenotype and disease term lists.

In [12]:
ALL_TERMS = PHENOTYPE_TERMS + DISEASE_TERMS

rows = []
for term in ALL_TERMS:
    mondo_hits = _call_ols4(term, ["MONDO"], exact=True) or _call_ols4(term, ["MONDO"], exact=False)
    hp_hits    = _call_ols4(term, ["HP"],    exact=True) or _call_ols4(term, ["HP"],    exact=False)

    top_mondo = mondo_hits[0] if mondo_hits else {}
    top_hp    = hp_hits[0]    if hp_hits    else {}

    mondo_iri  = top_mondo.get("iri", "")
    hp_iri     = top_hp.get("iri", "")
    mondo_curie = uri_to_curie(mondo_iri) if mondo_iri else None
    hp_curie    = uri_to_curie(hp_iri)    if hp_iri    else None

    if mondo_curie:
        winner_curie    = mondo_curie
        winner_label    = top_mondo.get("label")
        winner_ontology = "MONDO"
        biolink_category = "biolink:Disease"
    elif hp_curie:
        winner_curie    = hp_curie
        winner_label    = top_hp.get("label")
        winner_ontology = "HP"
        biolink_category = "biolink:PhenotypicFeature"
    else:
        winner_curie    = None
        winner_label    = None
        winner_ontology = None
        biolink_category = None

    rows.append({
        "term":             term,
        "MONDO_curie":      mondo_curie,
        "MONDO_label":      top_mondo.get("label"),
        "HP_curie":         hp_curie,
        "HP_label":         top_hp.get("label"),
        "winner_curie":     winner_curie,
        "winner_label":     winner_label,
        "winner_ontology":  winner_ontology,
        "biolink_category": biolink_category,
    })

df_winner = pd.DataFrame(rows)
df_winner

[error] giving up on text='irritable bowel syndrome' after 5 attempts


,term,MONDO_curie,MONDO_label,HP_curie,HP_label,winner_curie,winner_label,winner_ontology,biolink_category
0,microcephaly,MONDO:0001149,microcephaly,HP:0011451,Primary microcephaly,MONDO:0001149,microcephaly,MONDO,biolink:Disease
1,cleft palate,MONDO:0016064,cleft palate,HP:0009099,Median cleft palate,MONDO:0016064,cleft palate,MONDO,biolink:Disease
2,short stature,MONDO:0014785,"microcephaly, short stature, and impaired gluc...",HP:0003510,Severe short stature,MONDO:0014785,"microcephaly, short stature, and impaired gluc...",MONDO,biolink:Disease
3,healthy,MONDO:0009194,immunodeficiency 32B,HP:0001394,Cirrhosis,MONDO:0009194,immunodeficiency 32B,MONDO,biolink:Disease
4,acid reflux,MONDO:0007186,gastroesophageal reflux disease,HP:0002020,Gastroesophageal reflux,MONDO:0007186,gastroesophageal reflux disease,MONDO,biolink:Disease
5,Crohn's disease,MONDO:0005011,Crohn disease,HP:0100280,Crohn's disease,MONDO:0005011,Crohn disease,MONDO,biolink:Disease
6,ulcerative colitis,MONDO:0005101,ulcerative colitis,HP:0100279,Ulcerative colitis,MONDO:0005101,ulcerative colitis,MONDO,biolink:Disease
7,colorectal cancer,MONDO:0005575,colorectal cancer,HP:6000221,Positive stool occult blood test,MONDO:0005575,colorectal cancer,MONDO,biolink:Disease
8,type 2 diabetes,MONDO:0005148,type 2 diabetes mellitus,HP:0005978,Type II diabetes mellitus,MONDO:0005148,type 2 diabetes mellitus,MONDO,biolink:Disease
9,irritable bowel syndrome,MONDO:0005052,irritable bowel syndrome,None,None,MONDO:0005052,irritable bowel syndrome,MONDO,biolink:Disease
